# 2 · From a MIDI file to a new melody

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sem-16161d/GreasyPidginTutorial/blob/main/EN/jupyter/markov-book-en/02_from_midi_to_melody.ipynb)

Now the symbols are **notes**. The plan:

1. choose a Scarlatti sonata (default: K.555)
2. let GreasyPidgin read it
3. learn which note tends to follow which
4. generate a new melody
5. write it to a MIDI file you can download

In [ ]:
# Setup -- runs by itself (about 20 seconds the first time)
import importlib.util, pathlib, subprocess, sys, urllib.request

GP_REF        = "main"   # which version of GreasyPidgin to use (a git branch or tag)
TUTORIAL_REF  = "main"   # which version of this tutorial's helper file to use
GP_REPO       = "sem-16161d/GreasyPidgin"
TUTORIAL_REPO = "sem-16161d/GreasyPidginTutorial"

if importlib.util.find_spec("GreasyPidgin") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                    f"git+https://github.com/{GP_REPO}.git@{GP_REF}"], check=True)

if not pathlib.Path("workshop_helpers.py").exists():
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/{TUTORIAL_REPO}/{TUTORIAL_REF}/EN/jupyter/markov-book-en/workshop_helpers.py",
        "workshop_helpers.py")

import random
import matplotlib.pyplot as plt
from GreasyPidgin import MarkovChain, MarkovRule, MarkovRules
import workshop_helpers as wh

print("Ready!")

## Step 1: choose a sonata

The training material is one of Domenico Scarlatti's 555 keyboard sonatas (Kirkpatrick numbering, K.1 to K.555).
The default is the last one: **K.555**.

**Change the number in the cell below** (1 to 555) and run the cell to train on a different sonata.
The sonatas have two hands; if several notes start together, only the highest one is kept.
A MIDI file of your own? See "Try it" at the very bottom. (MIDI files: © Sankey, according to the file names.)

In [ ]:
SONATA = 555      # <- change this number (1 to 555) to train on a different sonata

midi_path = wh.get_scarlatti(SONATA, TUTORIAL_REPO, TUTORIAL_REF)

## Step 2: read it with GreasyPidgin

Behind the scenes, `Gesture.fromMidiFile` turns every MIDI note into a **Phonon** —
GreasyPidgin's note object — and collects them in a **Gesture**.
We take pitches and rhythms out of it and keep them in a small `Melody` object.

In [ ]:
melody = wh.load_melody(midi_path)
print(melody)
wh.plot_melody(melody)
plt.show()

In [ ]:
wh.listen(midi_path)

`melody.pitches` are MIDI note numbers (60 = middle C, written `c4`; `s` means sharp and `f` flat, so `cs4` is C#4 and `bf3` is Bb3), `melody.deltas` are the times between
note starts in beats (1.0 = one quarter note), `melody.bpm` is the tempo.

In [ ]:
print(melody.pitches[:12])
print(melody.deltas[:12])
print(melody.bpm)

## Step 3: learn the chain

`ORDER` is the memory: how many previous notes decide the next one.

In [ ]:
ORDER = 2

pitch_chain = MarkovChain()
pitch_chain.generateRulesFromList(melody.pitches, order=ORDER)
print(f"{len(pitch_chain.rules)} rules learned. The first three:")

for state in pitch_chain.rules.elements[:3]:
    names = " ".join(wh.pitch_name(p) for p in state)
    print(f"  after  {names:8s} ->", pitch_chain.rules[state])

## Step 4: generate

We start the new melody with the first notes of the original and let the chain continue.

In [ ]:
N_NOTES = 32

start = tuple(melody.pitches[:ORDER])
new_pitches = pitch_chain.getNElements(seed=start, n=N_NOTES)

print(" ".join(wh.pitch_name(p) for p in new_pitches))

## Step 5: a rhythm, and a MIDI file

For now the new melody borrows the rhythm of the original (repeating it as often as needed).
The next chapter teaches the chain rhythm, too.

We build a **Gesture** from parallel lists: one list for pitches, one for the time between
starts, one for durations, plus a dynamic and a tempo.

In [ ]:
from GreasyPidgin import Gesture

rhythm = [melody.deltas[i % len(melody.deltas)] for i in range(len(new_pitches))]

gesture = Gesture.fromLists(
    size        = len(new_pitches),
    startOffset = 0,
    startDeltas = [0] + rhythm[:-1],           # time before each note starts
    durations   = [r * 0.9 for r in rhythm],   # slightly detached
    pitches     = new_pitches,
    dynamics    = 80,                          # MIDI velocity
    timeUnit    = "b",                         # all times are in beats
    bpm         = melody.bpm,
)

gesture.toMidi("new_melody.mid", bpm=melody.bpm)

In [ ]:
wh.plot_notes(new_pitches, rhythm, title="The new melody", color="C1")
plt.show()
wh.listen("new_melody.mid")

Happy with it? Save the MIDI file to your computer:

In [ ]:
wh.download("new_melody.mid")

## A different approach: intervals instead of notes

So far the chain has learned **absolute notes**: "after `c5` comes `ef5`". It only plays notes that occurred in the
original, always in the same register. Now let it learn **intervals** instead: the distance between two consecutive
notes in semitones (`+2` = a major second up, `-3` = a minor third down).
An interval like "two semitones up" occurs in many places, whatever the note. So the chain learns the *shape* of the
melody, not its pitches.
Intervals are the same everywhere, so the chain can learn from **several sonatas at once**: list their numbers in
`SONATAS` below (a single number works too). The sonatas are simply joined one after another.

In [ ]:
SONATAS = [555, 554, 553]    # <- one sonata or several, e.g. [100] or [100, 101, 102]

intervals = []
all_pitches = []
for number in SONATAS:
    m = wh.load_melody(wh.get_scarlatti(number, TUTORIAL_REPO, TUTORIAL_REF))
    intervals += [b - a for a, b in zip(m.pitches, m.pitches[1:])]
    all_pitches += m.pitches
print(intervals[:12])      # semitone steps between consecutive notes

interval_chain = MarkovChain()
interval_chain.generateRulesFromList(intervals, order=ORDER)
print(f"Learned {len(interval_chain.rules)} rules from {len(SONATAS)} sonata(s) "
      f"(the note chain on K.{SONATA} alone had {len(pitch_chain.rules)}).")

To generate, we start on the first note of the original and add the learned intervals one after another.
If the melody leaves the range of the original, we move it back by octaves.

In [ ]:
low, high = min(all_pitches), max(all_pitches)      # range of all training sonatas

new_intervals = interval_chain.getNElements(seed=tuple(intervals[:ORDER]), n=N_NOTES)

pitch = melody.pitches[0]            # start note: the first note of the original
interval_pitches = [pitch]
for step in new_intervals[:N_NOTES - 1]:
    pitch += step
    while pitch > high:              # stay within the range of the original
        pitch -= 12
    while pitch < low:
        pitch += 12
    interval_pitches.append(pitch)

print(" ".join(wh.pitch_name(p) for p in interval_pitches))

In [ ]:
wh.render_midi(interval_pitches, rhythm, melody.bpm, "new_melody_intervals.mid")
wh.plot_notes(interval_pitches, rhythm, title="The new melody (intervals)", color="C2")
plt.show()
wh.listen("new_melody_intervals.mid")

Listen to both melodies one after the other. The note chain stays with the tones and turns of the original. The interval
chain puts its movements together anew at arbitrary places, and can hit tones that never occurred in the original.

In [ ]:
wh.download("new_melody_intervals.mid")

## Chords: keep all the notes

So far every chord was reduced to its highest note, and the rhythm was borrowed from the original.
Now the chain hears **everything**. A chord is just notes that start at the same time, written from the lowest note up,
and the chain treats them like any other series of notes.

Every note is one symbol of two numbers: `(interval, gap)`.

- `interval` is the distance in semitones from the previous note, as before.
- `gap` is the time since the previous note started, in beats. **`gap = 0` is allowed:** the note starts together with the
  previous one, so the two form a chord.

The chain learns the rhythm together with the intervals.

In [ ]:
import io, contextlib

chord_symbols = []           # one symbol per note: (interval, gap)
used_pitches = []

for number in SONATAS:
    with contextlib.redirect_stdout(io.StringIO()):     # Gesture.fromMidiFile prints every note
        g = Gesture.fromMidiFile(wh.get_scarlatti(number, TUTORIAL_REPO, TUTORIAL_REF))
    # all notes, chords included, as (start in beats, pitch); inside a chord from low to high
    notes = sorted({(round(p.startTimeBeats, 3), round(q)) for p in g.children for q in p.allPitches})
    for (t0, a), (t1, b) in zip(notes, notes[1:]):
        chord_symbols.append((b - a, round(t1 - t0, 1)))     # (interval, gap); gap 0 = same time as the note before
    used_pitches += [q for _, q in notes]

print(chord_symbols[:8])

chord_chain = MarkovChain()
chord_chain.generateRulesFromList(chord_symbols, order=ORDER)
in_chords = sum(1 for _, gap in chord_symbols if gap == 0)
print(f"{len(chord_symbols)} notes, {in_chords} of them inside chords. {len(chord_chain.rules)} rules learned.")

To generate, the chain starts from a random learned state and adds interval after interval, moving back into the range
by octaves as before. Notes with `gap = 0` sound together.

In [ ]:
N_NOTES = 200
low, high = min(used_pitches), max(used_pitches)

seed = random.choice(chord_chain.rules.elements)           # a random starting state
new_symbols = chord_chain.getNElements(seed=seed, n=N_NOTES)

pitch = melody.pitches[0]
chord_pitches, gaps = [pitch], [0]
for step, gap in new_symbols[:N_NOTES - 1]:
    pitch += step
    while pitch > high:                       # stay within the range of the originals
        pitch -= 12
    while pitch < low:
        pitch += 12
    chord_pitches.append(pitch)
    gaps.append(gap)

groups = []                                   # print chords as c4+e4+g4
for p, gap in zip(chord_pitches, gaps):
    if gap == 0 and groups:
        groups[-1].append(wh.pitch_name(p))
    else:
        groups.append([wh.pitch_name(p)])
print("  ".join("+".join(g) for g in groups))

In [ ]:
# each note sounds until the next chord starts
durations = [next((g for g in gaps[i + 1:] if g > 0), 1.0) * 0.9 for i in range(len(gaps))]

chord_gesture = Gesture.fromLists(
    size        = len(chord_pitches),
    startOffset = 0,
    startDeltas = gaps,                       # 0 = same time as the note before: a chord
    durations   = durations,
    pitches     = chord_pitches,
    dynamics    = 80,
    timeUnit    = "b",
    bpm         = melody.bpm,
)
chord_gesture.toMidi("new_melody_chords.mid", bpm=melody.bpm)

wh.plot_notes(chord_pitches, gaps[1:] + [1.0], durations,    # plot_notes wants the time AFTER each note
              title="The new melody (with chords)", color="C3")
plt.show()
wh.listen("new_melody_chords.mid")

Chords appear wherever the chain draws a `gap` of `0`. The rhythm now comes from the chain as well, not from the original.

In [ ]:
wh.download("new_melody_chords.mid")

## Higher order: a longer memory

So far the chain looked at the last `ORDER` notes. With a **higher order** it remembers a longer stretch of music, so the
melody follows the original more closely. The price: the higher the order, the fewer choices are left, and at some point the
chain can only repeat the original. The table shows how much of a new melody is copied from the original for each order
(share of runs of 4 notes). The cell then generates with `HIGH_ORDER`.

In [ ]:
HIGH_ORDER = 3          # memory: how many previous notes decide the next one
N_NOTES = 200

for order in (1, 2, 3, 4):             # how much of the original does each order copy?
    chain = MarkovChain()
    chain.generateRulesFromList(chord_symbols, order=order)
    new = chain.getNElements(seed=random.choice(chain.rules.elements), n=N_NOTES)
    print(f"order {order}: {len(chain.rules):5d} rules, {wh.copy_ratio(chord_symbols, new):4.0%} copied")

high_chain = MarkovChain()
high_chain.generateRulesFromList(chord_symbols, order=HIGH_ORDER)
high_symbols = high_chain.getNElements(seed=random.choice(high_chain.rules.elements), n=N_NOTES)

In [ ]:
def render_chords(pitches, gaps, durations, path, title, color):
    gesture = Gesture.fromLists(
        size=len(pitches), startOffset=0, startDeltas=gaps, durations=durations,
        pitches=pitches, dynamics=80, timeUnit="b", bpm=melody.bpm,
    )
    gesture.toMidi(path, bpm=melody.bpm)
    wh.plot_notes(pitches, gaps[1:] + [1.0], durations, title=title, color=color)
    plt.show()
    wh.listen(path)


pitch = melody.pitches[0]
high_pitches, gaps = [pitch], [0]
for step, gap in high_symbols[:N_NOTES - 1]:
    pitch += step
    while pitch > high:                       # stay within the range of the originals
        pitch -= 12
    while pitch < low:
        pitch += 12
    high_pitches.append(pitch)
    gaps.append(gap)

durations = [next((g for g in gaps[i + 1:] if g > 0), 1.0) * 0.9 for i in range(len(gaps))]
render_chords(high_pitches, gaps, durations, "new_melody_high_order.mid", "The new melody (higher order)", "C4")

In [ ]:
wh.download("new_melody_high_order.mid")

## Independent chains: pitch, gap and duration apart

So far one symbol carried everything: `(interval, gap)`. Now three **separate chains** learn the intervals, the gaps (time
between starts, `0` = chord) and the durations, each with the memory `HIGH_ORDER`. They know nothing about each other. Pitch and rhythm
are shuffled independently, so combinations appear that never occurred in the original.

In [ ]:
interval_list, gap_list, duration_list = [], [], []

for number in SONATAS:
    with contextlib.redirect_stdout(io.StringIO()):
        g = Gesture.fromMidiFile(wh.get_scarlatti(number, TUTORIAL_REPO, TUTORIAL_REF))
    # all notes as (start in beats, pitch, duration); inside a chord from low to high
    notes = sorted({(round(p.startTimeBeats, 3), round(q), max(0.1, round(p.durationBeats, 1)))
                    for p in g.children for q in p.allPitches})
    for (t0, a, _), (t1, b, d) in zip(notes, notes[1:]):
        interval_list.append(b - a)
        gap_list.append(round(t1 - t0, 1))
        duration_list.append(d)

interval_chain3, gap_chain, duration_chain = MarkovChain(), MarkovChain(), MarkovChain()
interval_chain3.generateRulesFromList(interval_list, order=HIGH_ORDER)
gap_chain.generateRulesFromList(gap_list, order=HIGH_ORDER)
duration_chain.generateRulesFromList(duration_list, order=HIGH_ORDER)
print(f"Rules learned: intervals {len(interval_chain3.rules)}, gaps {len(gap_chain.rules)}, durations {len(duration_chain.rules)}.")

In [ ]:
N_NOTES = 200

def draw(chain):                              # n elements from a random starting state
    return chain.getNElements(seed=random.choice(chain.rules.elements), n=N_NOTES)

new_intervals = draw(interval_chain3)
new_gaps      = draw(gap_chain)
new_durations = draw(duration_chain)

pitch = melody.pitches[0]
indep_pitches, gaps = [pitch], [0]
for step, gap in zip(new_intervals[:N_NOTES - 1], new_gaps):
    pitch += step
    while pitch > high:
        pitch -= 12
    while pitch < low:
        pitch += 12
    indep_pitches.append(pitch)
    gaps.append(gap)

render_chords(indep_pitches, gaps, new_durations[:N_NOTES], "new_melody_independent.mid", "The new melody (independent chains)", "C5")

In [ ]:
wh.download("new_melody_independent.mid")

## Again, again!

Go back to **Step 4** and run the cells below it again: every run is a new melody.

**Try it:**

- Change `ORDER` to `1` or `3` in Step 3 (and run Step 3 and 4 again). Does the new melody still
  sound like the original?
- Make it longer (`N_NOTES = 100`).
- Start the interval melody on a different note: set `pitch = 60` instead of `pitch = melody.pitches[0]`.
  It has the same shape, just shifted.
- Train on more sonatas: put more numbers into `SONATAS`, e.g. `SONATAS = [100, 101, 102, 103]`, and run the interval
  and chord cells again.
- Chords: change `ORDER` and compare how many chords you get. Or start the chord melody on another note: set `pitch = 48` instead of `pitch = melody.pitches[0]` in the chord cell.
- Change `HIGH_ORDER` to `1`, `2` or `6`. What happens to the copied share in the table, and to the independent chains below?
- Pick a different sonata: change `SONATA` in Step 1 and run all cells from Step 1 again.
- Upload a MIDI file of your own: replace the `midi_path = ...` line in Step 1 with `midi_path = wh.choose_midi()`.
- Change `0.9` in Step 5 to `1.0` (legato) or `0.4` (staccato).